# Stakeholder Dashboard Challenge

---

## Overview

You have access to 2+ years of real market data (stock prices from Yahoo Finance). Your organization is building a product and wants a dashboard, but stakeholders disagree on what it should show.

**Head of Strategy:**
> "We're building for institutional investors. Show volatility by asset, correlation matrices, capital efficiency metrics, and tail risk. Make it serious."

**Product Lead:**
> "Actually, our retail users want simplicity. Show moving averages, momentum, and 'is now a good time to buy?' Make it visual and accessible."

**Engineering:**
> "Whatever you build, pull live data from Yahoo Finance with a fallback CSV if the API fails."

---

## Your Assignment

Build a professional dashboard that serves **one specific stakeholder persona** and present your design choices.

**Deliverable:**
1. A functional dashboard (notebook cells with visualizations)
2. A presentation of your findings covering:
   - Who this dashboard is designed for (audience persona)
   - Which 3-5 metrics you chose and why they matter for this audience
   - What business decision or action they would take based on what they see
   - Why you rejected other metrics/audiences
3. **Share your work via GitHub** (create a repo, push your notebook and analysis)

**Technical Requirements:**
- Load data via yfinance (with CSV fallback if the API fails)
- Clean and validate the data
- Calculate relevant metrics
- Create professional visualizations (subplots, clear labels, appropriate scales)

---

In [ ]:
%pip install yfinance

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries loaded")

In [ ]:
# Load market data: try live first, fall back to CSV
tickers = ['AAPL', 'MSFT', 'GOOGL', 'NVDA', 'TSLA', 'META', 'AMZN', 'BRK-B']
start_date = (datetime.now() - timedelta(days=730)).strftime('%Y-%m-%d')
end_date = datetime.now().strftime('%Y-%m-%d')

try:
    data = yf.download(tickers, start=start_date, end=end_date, progress=False)['Close']
    print(f"✓ Live data loaded from Yahoo Finance ({len(data)} trading days)")
except:
    data = pd.read_csv('data/week1_market_data_fallback.csv', index_col=0, parse_dates=True)
    print(f"✓ Fallback data loaded from CSV ({len(data)} trading days)")

print(f"  Assets: {list(data.columns)}")
print(f"  Period: {data.index[0].date()} to {data.index[-1].date()}")
print(f"  Data shape: {data.shape}")

## Data Exploration & Analysis

Clean the data, calculate metrics, and build your dashboard below.

In [ ]:
# Your analysis and dashboard code here
data.head()

In [ ]:
print("\nMissing values:")
print(data.isna().sum())


In [ ]:
# Calculate daily returns: (Price_today - Price_yesterday) / Price_yesterday
returns = data.pct_change().dropna()

# Convert to percentage
returns_pct = returns * 100

print("Daily Returns (first 5 days):")
print(returns_pct.head())
print(f"\nDaily Returns Statistics:")
print(returns_pct.describe().round(3))

In [ ]:
# Volatility = Standard deviation of returns
daily_volatility = returns.std()

# Annualize volatility (there are ~252 trading days per year)
annual_volatility = daily_volatility * np.sqrt(252)

# Create summary table
volatility_df = pd.DataFrame({
    'Daily Volatility (%)': daily_volatility * 100,
    'Annual Volatility (%)': annual_volatility * 100,
    'Avg Daily Return (%)': returns_pct.mean(),
    'Max Daily Return (%)': returns_pct.max(),
    'Min Daily Return (%)': returns_pct.min(),
})

print("\n" + "="*70)
print("VOLATILITY & RETURN ANALYSIS")
print("="*70)
print(volatility_df.round(2))

print(f"\nInterpretation:")
print(f"  - Higher volatility = more risky (bigger price swings)")
print(f"  - Lower volatility = more stable (smaller price swings)")
print(f"  - Annual volatility of 30% = typical for tech stocks")

In [ ]:
 # Normalize prices to 100 at start (for easier comparison)
normalized_prices = (data / data.iloc[0] * 100)

# Plot normalized prices
fig, ax = plt.subplots(figsize=(14, 6))

for stock in tickers:
    ax.plot(normalized_prices.index, normalized_prices[stock], linewidth=2, label=stock)

ax.set_xlabel('Date', fontsize=11)
ax.set_ylabel('Price (Indexed to 100)', fontsize=11)
ax.set_title('Stock Price Performance (2 Years)', fontsize=12, fontweight='bold')
ax.legend(fontsize=10, loc='best')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('stock_prices.png', dpi=150, bbox_inches='tight')
plt.show()

print("Chart saved: stock_prices.png")
print(f"\nInterpretation:")
best_performer = normalized_prices.iloc[-1].idxmax()
worst_performer = normalized_prices.iloc[-1].idxmin()
print(f"  Best performer: {best_performer} (${normalized_prices.iloc[-1][best_performer]:.0f} vs starting $100)")
print(f"  Worst performer: {worst_performer} (${normalized_prices.iloc[-1][worst_performer]:.0f} vs starting $100)")

In [ ]:
# Calculate moving averages for Apple
stock_to_analyze = 'AAPL'
ma_short = data[stock_to_analyze].rolling(window=20).mean()  # 20-day MA
ma_long = data[stock_to_analyze].rolling(window=200).mean()  # 200-day MA

# Plot with moving averages
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(data.index, data[stock_to_analyze], label='Daily Price', linewidth=1, alpha=0.7)
ax.plot(ma_short.index, ma_short, label='20-Day Moving Average', linewidth=2, color='orange')
ax.plot(ma_long.index, ma_long, label='200-Day Moving Average', linewidth=2, color='red')

ax.set_xlabel('Date', fontsize=11)
ax.set_ylabel('Price ($)', fontsize=11)
ax.set_title(f'{stock_to_analyze} Price with Moving Averages', fontsize=12, fontweight='bold')
ax.legend(fontsize=10, loc='best')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('moving_averages.png', dpi=150, bbox_inches='tight')
plt.show()

print("Chart saved: moving_averages.png")
print(f"\nMoving Average Interpretation:")
print(f"  20-day MA: Shows short-term trend (1 month)")
print(f"  200-day MA: Shows long-term trend (1 year)")
print(f"  Golden Cross: 20-day MA crosses above 200-day = bullish signal")
print(f"  Death Cross: 20-day MA crosses below 200-day = bearish signal")

In [ ]:
# Calculate rolling volatility (30-day window)
rolling_volatility = returns.rolling(window=30).std() * np.sqrt(252) * 100

# Plot rolling volatility
fig, ax = plt.subplots(figsize=(14, 6))

for stock in tickers:
    ax.plot(rolling_volatility.index, rolling_volatility[stock], linewidth=2, label=stock)

ax.set_xlabel('Date', fontsize=11)
ax.set_ylabel('Annualized Volatility (%)', fontsize=11)
ax.set_title('Rolling 30-Day Volatility', fontsize=12, fontweight='bold')
ax.legend(fontsize=10, loc='best')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('rolling_volatility.png', dpi=150, bbox_inches='tight')
plt.show()

print("Chart saved: rolling_volatility.png")
print(f"\nKey Observations:")
print(f"  - Volatility changes over time (not constant)")
print(f"  - Market crises cause volatility spikes")
print(f"  - Current volatility useful for traders")

In [ ]:
# Calculate correlation between daily returns
correlation = returns.corr()

print("\nCorrelation Matrix (Daily Returns):")
print(correlation.round(3))

# Visualize as heatmap
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(correlation, annot=True, fmt='.2f', cmap='coolwarm', center=0, 
            square=True, ax=ax, cbar_kws={'label': 'Correlation'})
ax.set_title('Stock Return Correlations', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print("Chart saved: correlation_heatmap.png")
print(f"\nInterpretation:")
print(f"  Correlation = 1.0: Move perfectly together (no diversification benefit)")
print(f"  Correlation = 0.0: Move independently (good diversification)")
print(f"  Correlation < 0: Move opposite (excellent diversification)")

## Presentation of Findings

### 1. Audience Persona
This dashboard is designed for a **self-directed retail investor** who wants to compare large U.S. stocks while balancing growth and risk.

### 2. Metrics Rationale

**a. Two-year normalized price performance**  
Stock prices are normalized to 100 so stocks with different prices can be compared directly. It shows which stocks performed best over the same period.

**b. 20-day and 200-day moving averages (AAPL)**  
The 20-day MA shows the short-term trend, while the 200-day MA shows the long-term trend. This helps investors understand whether the stock is currently strong or weak relative to its recent trend.

**c. 30-day rolling volatility**  
Rolling volatility shows how much each stock's returns fluctuate over time. Higher volatility means higher risk and larger price swings.

**d. Return correlation**  
Correlation shows how closely stocks move together and helps evaluate diversification. Lower correlation can help reduce portfolio risk.

### 3. Business Decision
The investor can use the dashboard to decide how to allocate or rebalance the portfolio. For example, GOOGL had strong two-year performance, while TSLA showed much higher volatility. BRK-B had lower volatility and a very low correlation with NVDA, so it could help diversify a technology-heavy portfolio.

For AAPL, the moving averages can help the investor decide whether to hold, buy more, or wait for a stronger short-term trend.

### 4. Trade-offs
I focused on performance, trend, volatility, and correlation because they directly support a retail investor's decisions about return and risk. I excluded more advanced metrics such as Sharpe ratio, beta, VaR, and portfolio optimization because they would make the dashboard more complex for the target audience.

I also focused on a small group of large U.S. stocks rather than the entire market, which makes the dashboard easier to compare but limits how broadly the results can be applied.
